<a href="https://colab.research.google.com/github/Nethra910/audio-summarizer/blob/main/Audio_to_summirizer_using_huggingface_and_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio librosa soundfile huggingface_hub

In [ ]:
import torch
import gradio as gr
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextStreamer
)

In [ ]:
whisper_model = "openai/whisper-medium.en"

In [ ]:
speech_to_text = pipeline(
    task="automatic-speech-recognition",
    model = whisper_model,
    dtype = torch.float16,
    device = "cuda",
    chunk_length_s=30,
    return_timestamps=True

)

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(hf_token)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
from transformers import AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained( model_name )
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = quant_config,
    device_map = "auto"
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
def transcribe_audio(audio):
  result = speech_to_text(audio)
  return result["text"]

In [ ]:
def summarize_text(text):
  messages = [
    {
        "role": "system",
        "content": """
You are an expert audio transcript summarization assistant.

Your task is to summarize transcripts accurately and clearly.

Rules:
- Do not add information that is not present in the transcript.
- Keep the summary concise and easy to understand.
- Identify the most important ideas, facts, decisions, and conclusions.
- Remove unnecessary repetition and filler words.
- If the transcript is unclear, do not guess; mention that the information is unclear.
- Use clear headings and bullet points. give me the response in telugu language.
"""
    },
    {
        "role": "user",
        "content": f"""
Summarize the following transcript.

Give:
1. A short summary
2. Important points
3. Key conclusions

Transcript:

{text}
"""
    }
  ]
  inputs = tokenizer.apply_chat_template(
      messages,
      return_tensors = "pt",
      return_dict=True,
      add_generation_prompt=True
  ).to("cuda")
  streamer = TextStreamer(tokenizer)
  outputs = model.generate(
      **inputs,
      max_new_tokens = 2000,
      streamer=streamer
  )
  input_length = inputs["input_ids"].shape[-1]
  generated_tokens = outputs[0][input_length:]
  summary = tokenizer.decode(
      generated_tokens,
      skip_special_tokens=True
  )

  return summary



In [ ]:
def audio_summarizer(audio):
  if audio is None:
    return "Please upload an audio file."

  transcript = transcribe_audio(audio)
  summary = summarize_text(transcript)
  return f"""
TRANSCRIPT
==========

{transcript}


SUMMARY
=======

{summary}
"""

In [ ]:
view = gr.Interface(
    fn = audio_summarizer,
    inputs = gr.Audio(
        type = "filepath",
        label = "Upload Audio"
    ),
    outputs = gr.Markdown(
        label = "Audio Summarizer",
    )
)
view.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://672fedce77185d59bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processi

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 15 Sep 2026

You are an expert audio transcript summarization assistant.

Your task is to summarize transcripts accurately and clearly.

Rules:
- Do not add information that is not present in the transcript.
- Keep the summary concise and easy to understand.
- Identify the most important ideas, facts, decisions, and conclusions.
- Remove unnecessary repetition and filler words.
- If the transcript is unclear, do not guess; mention that the information is unclear.
- Use clear headings and bullet points. give me the response in telugu language.<|eot_id|><|start_header_id|>user<|end_header_id|>

Summarize the following transcript.

Give:
1. A short summary
2. Important points
3. Key conclusions

Transcript:

 Hello. I'm back. I'm back in San Francisco. As you may know, I have been traveling for the last ten weeks. So that's why I have not added more podcasts to the Effortless Eng